In [16]:
install.packages("table1")

Installing package into ‘/home/jupyter/.R/library’
(as ‘lib’ is unspecified)



In [17]:
library(bigrquery)
library(knitr)
library(tidyverse)
library(ggplot2)
library("table1")
library("IRdisplay")
bq_auth()

In [18]:
data <-bq_dataset_query(query="
SELECT *

 FROM `yhcr-prd-phm-bia-core.CB_1935_AK.src_bmbc_EYFSP_COMG03IMD`"
                       ,x="yhcr-prd-phm-bia-core.CB_1935_AK")

EYFSP3 <- bq_table_download(data) 

In [19]:
data1 <-bq_dataset_query(query="
select round(avg(count1),5)

from(

SELECT person_id,count(person_id) count1

 FROM `yhcr-prd-phm-bia-core.CB_1935_AK.src_bmbc_EYFSP_COMG03IMD`

 group by person_id)base"
                       ,x="yhcr-prd-phm-bia-core.CB_1935_AK")

EYFSP3AVG <- bq_table_download(data1) 

In [20]:
EYFSP3$Gender[(EYFSP3$Gender) == "F"] <- "Female"
EYFSP3$Gender[(EYFSP3$Gender) == "M"] <- "Male"
EYFSP3$Gender[(EYFSP3$Gender) == "U"] <- "Unknown"

In [21]:
display_jupyter <- function(x) {
  css <- system.file("table1_defaults_1.0/table1_defaults.css", package="table1")
  css <- paste(readLines(css), collapse="\n")
  x <- htmltools::tagList(htmltools::tags$style(css), htmltools::tags$div(class="Rtable1", x))
  IRdisplay::display_html(as.character(x))
}

In [22]:
label(EYFSP3$AcademicYear)      <- "Academic Year"
label(EYFSP3$IMD)      <- "Average IMD"

label(EYFSP3$FSMEligible)      <- "FSM Eligiblity"
label(EYFSP3$ethnic_group)      <- "Ethnic Group"
label(EYFSP3$Gender)      <- "Gender"

In [23]:
 x<-table1(~ EYFSP3$AcademicYear + EYFSP3$FSMEligible +EYFSP3$Gender+ EYFSP3$ethnic_group+ EYFSP3$IMD|EYFSP3$score,data=EYFSP3
          ,justify=c("left", "left", rep("center", 5)),format_number = TRUE
          ,caption = "Table presenting COMGO3 - Speaking results from the Early years foundation stage profile (EYFSP)  "
    
         )

In [24]:
display_jupyter(x)

,1 - Emerging(N=15742),2 - Expected level(N=47334),3 - Exceeded(N=12499),A - Exemption not assessed (N=180),No Data(N=23),Overall(N=75778)
Academic Year,,,,,,
2012/2013,3309 (21.0%),6733 (14.2%),1261 (10.1%),28 (15.6%),0 (0%),11331 (15.0%)
2013/2014,2926 (18.6%),6817 (14.4%),1599 (12.8%),18 (10.0%),0 (0%),11360 (15.0%)
2014/2015,2208 (14.0%),7094 (15.0%),1808 (14.5%),31 (17.2%),0 (0%),11141 (14.7%)
2015/2016,1977 (12.6%),6961 (14.7%),1968 (15.7%),0 (0%),23 (100%),10929 (14.4%)
2016/2017,1904 (12.1%),6828 (14.4%),2002 (16.0%),19 (10.6%),0 (0%),10753 (14.2%)
2017/2018,1783 (11.3%),6625 (14.0%),1906 (15.2%),52 (28.9%),0 (0%),10366 (13.7%)
2018/2019,1635 (10.4%),6276 (13.3%),1955 (15.6%),32 (17.8%),0 (0%),9898 (13.1%)
FSM Eligiblity,,,,,,
Yes,4096 (26.0%),9067 (19.2%),1398 (11.2%),36 (20.0%),6 (26.1%),14603 (19.3%)


In [26]:
OverallTrend1 <- EYFSP3 %>% select(1,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score)


PS1<- group_by(PS1, AcademicYear) %>% mutate(percent = (n/sum(n))*100)


#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,4)

PS1<- group_by(PS1, AcademicYear) %>% mutate(PctPerYear =(sum(percent)))
PS1 <- PS1 %>% select(1,3)

PS1 <- distinct(PS1)

In [28]:
OVERALL<-
ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = 1)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication",atop("COMGO3 - Speaking results from the Early years foundation stage profile (EYFSP)"), atop(italic("percentage of people achieveing atleast 'Expected' grade"), "")))) +


labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.5,  vjust = -1) +
                   #+
  theme_minimal()

In [29]:
OverallTrend1 <- EYFSP3 %>% select(1,2,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,ethnic_group) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,ethnic_group)


PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)


#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)

In [30]:
ethnicity <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = ethnic_group,color = ethnic_group)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication", atop(italic("percentage of people achieveing atleast 'Expected' grade"),
                                                                                  atop(italic("By ethnic group")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -2) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [31]:
OverallTrend1 <- EYFSP3 %>% select(1,4,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,Gender) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,Gender)


PS1<- group_by(PS1, AcademicYear,Gender) %>% mutate(percent = (n/sum(n))*100)
#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,Gender) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)
PS1 <-PS1<-unique(filter(PS1,Gender == "Female" |Gender == "Male" ))

In [32]:
gender <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = Gender,color = Gender)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication", atop(italic("percentage of people achieveing atleast 'Expected' grade"),
                                                                                  atop(italic("By Gender")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -1) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [33]:
OverallTrend1 <- EYFSP3 %>% select(1,10,5,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,IMD) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,IMD)


PS1<- group_by(PS1, AcademicYear,IMD) %>% mutate(percent = (n/sum(n))*100)
#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "2 - Expected level" |score == "3 - Exceeded" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,IMD) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)

In [34]:
 IMD <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = IMD,color = IMD)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication", atop(italic("percentage of people achieveing atleast 'Expected' grade"),
                                                                                  atop(italic("By IMD")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -1) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [36]:
library(grid)
#library(gridtext)
library(gridExtra)
plot <- list(OVERALL, gender, IMD, ethnicity)
pdf('1.2 Speech Language and Communication3.pdf',width=10, height=10)
  title <- "Plots for 1.2 Speech Language & Communication : COMGO3 Communication and Language - Speaking"
grid.text(title) 
plot
dev.off()
#

[[1]]

[[2]]

[[3]]

[[4]]


png 
  2